# RNN

In [1]:
import pandas as pd
import torch

In [2]:
df = pd.read_csv('data/100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
#tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'","")
    return text.split()


In [4]:
tokenize(df.iloc[2,0])

['who', 'wrote', 'to', 'kill', 'a', 'mockingbird']

In [5]:
#vocab

vocab = {'<unk>':0}

def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    merged_tokens = tokenized_question + tokenized_answer

    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

df.apply(build_vocab, axis=1)


print(vocab)

{'<unk>': 0, 'what': 1, 'is': 2, 'the': 3, 'capital': 4, 'of': 5, 'france': 6, 'paris': 7, 'germany': 8, 'berlin': 9, 'who': 10, 'wrote': 11, 'to': 12, 'kill': 13, 'a': 14, 'mockingbird': 15, 'harper-lee': 16, 'largest': 17, 'planet': 18, 'in': 19, 'our': 20, 'solar': 21, 'system': 22, 'jupiter': 23, 'boiling': 24, 'point': 25, 'water': 26, 'celsius': 27, '100': 28, 'painted': 29, 'mona': 30, 'lisa': 31, 'leonardo-da-vinci': 32, 'square': 33, 'root': 34, '64': 35, '8': 36, 'chemical': 37, 'symbol': 38, 'for': 39, 'gold': 40, 'au': 41, 'which': 42, 'year': 43, 'did': 44, 'world': 45, 'war': 46, 'ii': 47, 'end': 48, '1945': 49, 'longest': 50, 'river': 51, 'nile': 52, 'japan': 53, 'tokyo': 54, 'developed': 55, 'theory': 56, 'relativity': 57, 'albert-einstein': 58, 'freezing': 59, 'fahrenheit': 60, '32': 61, 'known': 62, 'as': 63, 'red': 64, 'mars': 65, 'author': 66, '1984': 67, 'george-orwell': 68, 'currency': 69, 'united': 70, 'kingdom': 71, 'pound': 72, 'india': 73, 'delhi': 74, 'discov

In [6]:
#to numerical indices

def text_to_indices(text, vocab):
    indexed_text = []

    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<unk>']) 

    return indexed_text

In [7]:
from torch.utils.data import Dataset, DataLoader


class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        num_question = text_to_indices(self.df.iloc[index]['question'], self.vocab) 
        num_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab) 

        return torch.tensor(num_question), torch.tensor(num_answer)

In [8]:
dataset = QADataset(df, vocab)

In [9]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [14]:
import torch.nn as nn 

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 50)
        self.RNN = nn.RNN(50,64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden, final = self.RNN(embedded_question)
        output = self.fc(final.squeeze(0))

        return output

In [69]:
learning_rate = 0.001
epochs = 20

model = SimpleRNN(len(vocab))

In [70]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [71]:
for epoch in range(epochs):
    total_loss = 0

    for question, answer in dataloader:
        optimizer.zero_grad()

        output =model(question)

        loss = criterion(output, answer.view(-1))

        loss.backward()

        optimizer.step()

        total_loss = total_loss + loss.item()

    print(f'Epoch: {epoch+1}, loss: {total_loss:.4f}')

Epoch: 1, loss: 524.7157
Epoch: 2, loss: 459.0313
Epoch: 3, loss: 382.6760
Epoch: 4, loss: 318.5989
Epoch: 5, loss: 266.6606
Epoch: 6, loss: 219.1331
Epoch: 7, loss: 174.6267
Epoch: 8, loss: 136.1166
Epoch: 9, loss: 103.7220
Epoch: 10, loss: 79.7618
Epoch: 11, loss: 61.4172
Epoch: 12, loss: 47.8752
Epoch: 13, loss: 38.1655
Epoch: 14, loss: 31.0884
Epoch: 15, loss: 25.3681
Epoch: 16, loss: 21.1289
Epoch: 17, loss: 17.7839
Epoch: 18, loss: 15.1119
Epoch: 19, loss: 12.9276
Epoch: 20, loss: 11.2154


In [97]:
def predict(model, question, threshold=0.2):
    model.eval() 

    with torch.no_grad():
        num_question = text_to_indices(question, vocab)
        question_tensor = torch.tensor(num_question).unsqueeze(0)
        output = model(question_tensor)

        probability = torch.nn.functional.softmax(output, dim=1)

        value, index = torch.max(probability, dim=1)

        if value.item() < threshold:
            return "I don't Know"
        
        print(list(vocab.keys())[index.item()])
        

In [98]:
predict(model, 'largest planet in our solar system')

jupiter


In [99]:
predict(model, 'What is the longest river in the world?')

nile


In [100]:
predict(model, 'Who developed the theory of relativity?')

albert-einstein


In [102]:
predict(model, "Who is the author of '1984'?'")

george-orwell


In [91]:
predict(model, 'Which planet is known as the Red Planet?')

mars
